<a href="https://colab.research.google.com/github/de-bayes/mysite/blob/main/point_estimates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from datetime import date
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

filepath = '/content/drive/MyDrive/Forecasting 2026/workflow/intermediary/output_fund.csv'
funds = pd.read_csv(filepath)


filepath = '/content/drive/MyDrive/Forecasting 2026/workflow/intermediary/experts.csv'
demo = pd.read_csv(filepath)


filepath = '/content/drive/MyDrive/Forecasting 2026/workflow/intermediary/kalshi_ids.csv'
kalshi_ids = pd.read_csv(filepath)

Mounted at /content/drive


In [ ]:
funds_small = (
    funds[["race_id", "pred_result"]]
    .rename(columns={"pred_result": "fundamentals"})
)

merged = (
    funds_small
    .merge(demo[["race_id", "cook", "sabato", "inside"]],
           on="race_id",
           how="left")
)

merged["poll_avg"] = np.nan
merged["enop"] = np.nan
merged["kalshi"] = np.nan

print(merged.head())

     race_id  fundamentals      cook    sabato    inside  poll_avg  enop  \
0  H2026AK00      3.153412  Likely R  Likely R  Likely R       NaN   NaN   
1  H2026AL01     43.146753       NaN       NaN       NaN       NaN   NaN   
2  H2026AL02    -12.885976       NaN       NaN       NaN       NaN   NaN   
3  H2026AL03     37.569038       NaN       NaN       NaN       NaN   NaN   
4  H2026AL04     55.746302       NaN       NaN       NaN       NaN   NaN   

   kalshi  
0     NaN  
1     NaN  
2     NaN  
3     NaN  
4     NaN  


#POLLING

In [ ]:
merged["fun_pol"] = merged["fundamentals"]

#EXPERT

In [ ]:
RANGES = {
    "Solid R":  (17, None),
    "Likely R": (9, 17),
    "Lean R":   (4, 9),
    "Tossup":   (-4, 4),
    "Lean D":   (-9, -4),
    "Likely D": (-17, -9),
    "Solid D":  (None, -17),
}

WEIGHTS = {"cook": 0.14, "sabato": 0.11, "inside": 0.08}

def closest_edge(x, low, high):
    """
    Return the closest boundary point of [low, high] (with open ends allowed),
    but ONLY when x is outside the interval.
    """
    if low is not None and x < low:
        return low
    if high is not None and x > high:
        return high
    return None  # inside (or open-ended and on the valid side)

def outlet_adjustment(orig_x, rating_label, weight):
    if pd.isna(rating_label) or pd.isna(orig_x):
        return 0.0
    rating_label = str(rating_label).strip()
    if rating_label not in RANGES:
        return 0.0  # unknown label -> no adjustment

    low, high = RANGES[rating_label]
    edge = closest_edge(orig_x, low, high)
    if edge is None:
        return 0.0  # inside range -> do nothing

    return weight * (edge - orig_x)

orig = merged["fun_pol"]

merged["fun_pol_exp"] = orig.copy()
for col, w in WEIGHTS.items():
    merged["fun_pol_exp"] += [
        outlet_adjustment(x, r, w) for x, r in zip(orig.values, merged[col].values)
    ]

print(merged.head())

     race_id  fundamentals      cook    sabato    inside  poll_avg  enop  \
0  H2026AK00      3.153412  Likely R  Likely R  Likely R       NaN   NaN   
1  H2026AL01     43.146753       NaN       NaN       NaN       NaN   NaN   
2  H2026AL02    -12.885976       NaN       NaN       NaN       NaN   NaN   
3  H2026AL03     37.569038       NaN       NaN       NaN       NaN   NaN   
4  H2026AL04     55.746302       NaN       NaN       NaN       NaN   NaN   

   kalshi    fun_pol  fun_pol_exp  
0     NaN   3.153412     5.082786  
1     NaN  43.146753    43.146753  
2     NaN -12.885976   -12.885976  
3     NaN  37.569038    37.569038  
4     NaN  55.746302    55.746302  


#KALSHI

In [ ]:
import requests
import datetime
import base64
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import padding

# Configuration
API_KEY_ID = '5c20a158-bff6-420f-9eec-43130a56d580'
PRIVATE_KEY_PATH = '/content/drive/MyDrive/Forecasting 2026/workflow/intermediary/ZD_api_key.txt'
BASE_URL = 'https://api.elections.kalshi.com'

def load_private_key(key_path):
    """Load the private key from file."""
    with open(key_path, "rb") as f:
        return serialization.load_pem_private_key(f.read(), password=None, backend=default_backend())

def create_signature(private_key, timestamp, method, path):
    """Create the request signature."""
    path_without_query = path.split('?')[0]
    message = f"{timestamp}{method}{path_without_query}".encode('utf-8')
    signature = private_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.DIGEST_LENGTH),
        hashes.SHA256()
    )
    return base64.b64encode(signature).decode('utf-8')

def get(private_key, api_key_id, path, base_url=BASE_URL):
    """Make an authenticated GET request to the Kalshi API."""
    timestamp = str(int(datetime.datetime.now().timestamp() * 1000))
    signature = create_signature(private_key, timestamp, "GET", path)

    headers = {
        'KALSHI-ACCESS-KEY': api_key_id,
        'KALSHI-ACCESS-SIGNATURE': signature,
        'KALSHI-ACCESS-TIMESTAMP': timestamp
    }

    return requests.get(base_url + path, headers=headers)

private_key = load_private_key(PRIVATE_KEY_PATH)

In [ ]:
import time
import threading
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# assumes you already defined:
# API_KEY_ID, BASE_URL, private_key, create_signature(...)

# ---------------------------
# 1) robust per-thread session
# ---------------------------
def _make_session():
    s = requests.Session()
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        status=6,
        backoff_factor=0.6,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

_tls = threading.local()

def _session():
    if not hasattr(_tls, "s"):
        _tls.s = _make_session()
    return _tls.s

# ---------------------------
# 2) authenticated GET (params-safe)
# ---------------------------
def authed_get_json(private_key, api_key_id, path, params=None, base_url=BASE_URL, timeout=25):
    """
    Authenticated GET for Kalshi Trade API.
    Signature: timestamp + method + path (NO query params in signed string). :contentReference[oaicite:1]{index=1}
    """
    timestamp = str(int(time.time() * 1000))
    signature = create_signature(private_key, timestamp, "GET", path)

    headers = {
        "KALSHI-ACCESS-KEY": api_key_id,
        "KALSHI-ACCESS-SIGNATURE": signature,
        "KALSHI-ACCESS-TIMESTAMP": timestamp,
    }

    r = _session().get(base_url + path, headers=headers, params=params, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"GET {r.url} -> {r.status_code}: {r.text[:300]}")
    return r.json()

# ---------------------------
# 3) markets-by-event (paged)
# ---------------------------
def list_markets_for_event(event_ticker, page_limit=1000):
    out = []
    params = {"event_ticker": event_ticker, "limit": page_limit}

    while True:
        j = authed_get_json(private_key, API_KEY_ID, "/trade-api/v2/markets", params=params)
        markets = j.get("markets", [])
        for m in markets:
            m["event_ticker"] = event_ticker
        out.extend(markets)

        cursor = j.get("cursor") or j.get("next_cursor") or j.get("next")
        if cursor:
            params["cursor"] = cursor
            continue

        if j.get("has_more") is True and not cursor:
            raise RuntimeError("API says has_more=true but returned no cursor.")
        break

    return out

# ---------------------------
# 4) orderbook-by-market
# ---------------------------
def fetch_orderbook(market_ticker, depth=50):
    j = authed_get_json(
        private_key, API_KEY_ID,
        f"/trade-api/v2/markets/{market_ticker}/orderbook",
        params={"depth": depth},
    )
    return {
        "market_ticker": market_ticker,
        "pulled_ts": int(time.time()),
        "orderbook_fp": j.get("orderbook_fp"),
        "orderbook": j.get("orderbook"),
    }

# ---------------------------
# 5) main pipeline
# ---------------------------
def pull_all_orderbooks_authed(kalshi_ids: pd.DataFrame, depth=50, workers_markets=12, workers_orderbooks=24):
    # unique event tickers
    event_tickers = kalshi_ids["event_ticker"].dropna().astype(str).unique().tolist()

    # mapping to bring race_id back in later
    race_event = kalshi_ids[["race_id", "event_ticker"]].dropna().drop_duplicates()

    # (A) fetch markets for each event (concurrent)
    all_markets = []
    with ThreadPoolExecutor(max_workers=workers_markets) as ex:
        futs = {ex.submit(list_markets_for_event, et): et for et in event_tickers}
        for fut in as_completed(futs):
            et = futs[fut]
            try:
                all_markets.extend(fut.result())
            except Exception as e:
                print(f"[WARN] markets failed for {et}: {e}")

    markets_df = pd.DataFrame(all_markets)
    if markets_df.empty:
        return markets_df, pd.DataFrame()

    # keep useful cols if present
    # (not required, but makes downstream easier)
    # markets_df columns vary; leave as-is.

    # (B) unique market tickers, then pull orderbooks (concurrent)
    market_tickers = markets_df["ticker"].dropna().astype(str).unique().tolist()

    orderbook_rows = []
    with ThreadPoolExecutor(max_workers=workers_orderbooks) as ex:
        futs = {ex.submit(fetch_orderbook, t, depth): t for t in market_tickers}
        for fut in as_completed(futs):
            t = futs[fut]
            try:
                orderbook_rows.append(fut.result())
            except Exception as e:
                print(f"[WARN] orderbook failed for {t}: {e}")

    orderbooks_df = pd.DataFrame(orderbook_rows)

    # (C) attach event_ticker + optional metadata
    meta_cols = [c for c in ["ticker", "event_ticker", "title", "subtitle", "status", "close_time"] if c in markets_df.columns]
    meta = (
        markets_df[meta_cols]
        .drop_duplicates(subset=["ticker"])
        .rename(columns={"ticker": "market_ticker"})
    )
    orderbooks_df = orderbooks_df.merge(meta, on="market_ticker", how="left")

    # (D) attach race_id via event_ticker mapping
    orderbooks_df = orderbooks_df.merge(race_event, on="event_ticker", how="left")

    return markets_df, orderbooks_df

# ---- run ----
markets_df, orderbooks_df = pull_all_orderbooks_authed(kalshi_ids, depth=50)

print("markets_df:", markets_df.shape)
print("orderbooks_df:", orderbooks_df.shape)

markets_df: (1049, 55)
orderbooks_df: (1049, 10)


In [ ]:
orderbooks_df.head()

,market_ticker,pulled_ts,orderbook_fp,orderbook,event_ticker,title,subtitle,status,close_time,race_id
0,KXHOUSERACE-AL03-26-D,1771789276,"{'no_dollars': [['0.0100', '500.00'], ['0.0200...","{'no': [[1, 500], [2, 200], [10, 46], [37, 634...",KXHOUSERACE-AL03-26,Will Democratic win the House race for AL-03?,::,active,2027-11-03T15:00:00Z,H2026AL03
1,KXHOUSERACE-AL02-26-D,1771789276,"{'no_dollars': [['0.0100', '47.00']], 'yes_dol...","{'no': [[1, 47]], 'no_dollars': [['0.0100', 47...",KXHOUSERACE-AL02-26,Will Democratic win the House race for AL-02?,::,active,2027-11-03T15:00:00Z,H2026AL02
2,KXHOUSERACE-AL07-26-R,1771789276,"{'no_dollars': [['0.0100', '500.00'], ['0.3700...","{'no': [[1, 500], [37, 640], [38, 500], [39, 1...",KXHOUSERACE-AL07-26,Will Republican win the House race for AL-07?,::,active,2027-11-03T15:00:00Z,H2026AL07
3,KXHOUSERACE-AL03-26-R,1771789276,"{'no_dollars': None, 'yes_dollars': [['0.0200'...","{'no': None, 'no_dollars': None, 'yes': [[2, 2...",KXHOUSERACE-AL03-26,Will Republican win the House race for AL-03?,::,active,2027-11-03T15:00:00Z,H2026AL03
4,KXHOUSERACE-AL02-26-R,1771789276,"{'no_dollars': [['0.0100', '500.00'], ['0.0200...","{'no': [[1, 500], [2, 200], [36, 651], [37, 50...",KXHOUSERACE-AL02-26,Will Republican win the House race for AL-02?,::,active,2027-11-03T15:00:00Z,H2026AL02


In [ ]:
markets_df.head()

,can_close_early,close_time,created_time,custom_strike,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,last_price,...,volume,volume_24h,volume_24h_fp,volume_fp,yes_ask,yes_ask_dollars,yes_bid,yes_bid_dollars,yes_sub_title,early_close_condition
0,True,2027-11-03T15:00:00Z,2026-01-06T01:12:46.978505Z,"{'Party': 'Republican', 'political_party': '92...",KXHOUSERACE-AL03-26,2027-01-04T17:00:00Z,2027-11-03T15:00:00Z,,False,96,...,149,0,0.00,149.00,100,1.0000,93,0.9300,Republican party,NaN
1,True,2027-11-03T15:00:00Z,2026-01-06T01:12:46.978505Z,"{'Party': 'Democratic', 'political_party': '57...",KXHOUSERACE-AL03-26,2027-01-04T17:00:00Z,2027-11-03T15:00:00Z,,False,3,...,202,0,0.00,202.00,3,0.0300,0,0.0000,Democratic party,NaN
2,True,2027-11-03T15:00:00Z,2026-01-06T01:12:45.676474Z,"{'Party': 'Republican', 'political_party': '92...",KXHOUSERACE-AL02-26,2027-01-04T17:00:00Z,2027-11-03T15:00:00Z,,False,5,...,332,0,0.00,332.00,4,0.0400,0,0.0000,Republican party,NaN
3,True,2027-11-03T15:00:00Z,2026-01-06T01:12:45.676474Z,"{'Party': 'Democratic', 'political_party': '57...",KXHOUSERACE-AL02-26,2027-01-04T17:00:00Z,2027-11-03T15:00:00Z,,False,94,...,1,0,0.00,1.00,99,0.9900,92,0.9200,Democratic party,NaN
4,True,2027-11-03T15:00:00Z,2026-01-06T01:12:50.691111Z,"{'Party': 'Republican', 'political_party': '92...",KXHOUSERACE-AL07-26,2027-01-04T17:00:00Z,2027-11-03T15:00:00Z,,False,3,...,406,0,0.00,406.00,3,0.0300,0,0.0000,Republican party,NaN


# KALSHI AGGREGATION (cool new way!!!, uses a market scoring system, kinda like pollster scorecards but for market health)

Liquidity-weighted probability aggregation:
- ranks each markets liquidity (volume, spread, OI) within its race type
- Computes composite score (0-1) using: 35% volume + 45% spread + 20% OI
- weights last_trade vs midpoint: liquid markets trust trades more (75%), thin markets trust order book (65%)
- Handles edge cases: one-sided books → NULL, no trades → NULL, stale trades → use nearest order

In [ ]:
# ==============================================================================
# LIQUIDITY-WEIGHTED AGGGREGATION (xyz)
# ==============================================================================

# Composite score weights (must sum to 1.0)
WEIGHT_VOLUME = 0.35   # 35% weight on trading volume
WEIGHT_SPREAD = 0.45   # 45% weight on bid-ask spread (tighter = better)
WEIGHT_OI = 0.20       # 20% weight on open interest

# Last trade weight bounds
LAST_TRADE_WEIGHT_MIN = 0.35  # Thin markets: 35% last_trade, 65% midpoint
LAST_TRADE_WEIGHT_MAX = 0.75  # Liquid markets: 75% last_trade, 25% midpoint

# EMA settings (for future use when historical data is available)
# EMA_ALPHA_MIN = 0.07  # Thin markets: slow EMA (7% new, 93% old)
# EMA_ALPHA_MAX = 0.30  # Liquid markets: fast EMA (30% new, 70% old)


def percentile_rank(value, all_values):
    """
    Compute percentile rank (0 to 1).
    Example: percentile_rank(75, [50, 60, 70, 80, 90]) = 0.6
    """
    if len(all_values) == 0:
        return 0.5
    count_below = sum(1 for v in all_values if v <= value)
    return count_below / len(all_values)


def compute_kalshi_probabilities(markets_df):
    """
    Apply liquidity-weighted aggregation to all markets.
    Returns DataFrame with: race_id, party, probability, and diagnostic columns.
    """
    df = markets_df.copy()

    # Extract party from yes_sub_title ("Democratic party" or "Republican party")
    df["party"] = df["yes_sub_title"].apply(
        lambda x: "R" if "republican" in str(x).lower() else ("D" if "democratic" in str(x).lower() else None)
    )
    df = df[df["party"].notna()].copy()

    # Attach race_id from kalshi_ids mapping
    race_event = kalshi_ids[["race_id", "event_ticker"]].dropna().drop_duplicates()
    df = df.merge(race_event, on="event_ticker", how="left")

    # Extract race_type from race_id (H=House, S=Senate, G=Governor)
    df["race_type"] = df["race_id"].str[0]

    # Convert numeric columns
    df["yes_bid"] = pd.to_numeric(df["yes_bid"], errors="coerce").fillna(0)
    df["yes_ask"] = pd.to_numeric(df["yes_ask"], errors="coerce").fillna(0)
    df["last_price"] = pd.to_numeric(df["last_price"], errors="coerce").fillna(0)
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0)
    df["open_interest"] = pd.to_numeric(df["open_interest"], errors="coerce").fillna(0)

    # Compute spread and midpoint (only valid if both bid AND ask exist)
    df["has_two_sided"] = (df["yes_bid"] > 0) & (df["yes_ask"] > 0)
    df["spread"] = np.where(df["has_two_sided"], df["yes_ask"] - df["yes_bid"], np.nan)
    df["midpoint"] = np.where(df["has_two_sided"], (df["yes_bid"] + df["yes_ask"]) / 2, np.nan)

    # Detect stale trades (last_price outside current bid-ask spread)
    df["is_stale"] = df["has_two_sided"] & (
        (df["last_price"] < df["yes_bid"]) | (df["last_price"] > df["yes_ask"])
    )

    # Compute percentile ranks WITHIN each race_type (House vs House, etc.)
    for race_type in df["race_type"].unique():
        mask = df["race_type"] == race_type
        subset = df[mask]

        volumes = subset["volume"].tolist()
        ois = subset["open_interest"].tolist()

        # Inverse spread: tighter spread = better = higher rank
        inv_spreads = []
        for _, row in subset.iterrows():
            if pd.notna(row["spread"]) and row["spread"] > 0:
                inv_spreads.append(1.0 / row["spread"])
            else:
                inv_spreads.append(0)

        vol_ranks = [percentile_rank(v, volumes) for v in subset["volume"]]
        spread_ranks = [percentile_rank(inv_spreads[i], inv_spreads) for i in range(len(inv_spreads))]
        oi_ranks = [percentile_rank(o, ois) for o in subset["open_interest"]]

        df.loc[mask, "volume_pct"] = vol_ranks
        df.loc[mask, "spread_pct"] = spread_ranks
        df.loc[mask, "oi_pct"] = oi_ranks

    # Composite liquidity score (0 to 1)
    df["composite"] = (
        WEIGHT_VOLUME * df["volume_pct"] +
        WEIGHT_SPREAD * df["spread_pct"] +
        WEIGHT_OI * df["oi_pct"]
    )

    # Last trade weight: maps composite (0-1) to weight (0.35-0.75)
    weight_range = LAST_TRADE_WEIGHT_MAX - LAST_TRADE_WEIGHT_MIN
    df["last_trade_weight"] = LAST_TRADE_WEIGHT_MIN + (weight_range * df["composite"])

    # Compute final probability
    def calc_prob(row):
        # NULL: no trades AND no volume
        if row["last_price"] == 0 and row["volume"] == 0:
            return np.nan

        # NULL: one-sided book (can't compute meaningful midpoint)
        if not row["has_two_sided"]:
            return np.nan

        ltw = row["last_trade_weight"]  # Last trade weight (0.35-0.75)
        mpw = 1.0 - ltw                  # Midpoint weight (0.25-0.65)

        # Handle stale trade: use nearest order instead of outdated last_price
        if row["is_stale"]:
            ltw = LAST_TRADE_WEIGHT_MIN  # Drop to floor weight
            mpw = 1.0 - ltw
            dist_bid = abs(row["last_price"] - row["yes_bid"])
            dist_ask = abs(row["last_price"] - row["yes_ask"])
            nearest = row["yes_bid"] if dist_bid < dist_ask else row["yes_ask"]
            return (ltw * nearest) + (mpw * row["midpoint"])
        else:
            # Normal case: weighted average of last_price and midpoint
            return (ltw * row["last_price"]) + (mpw * row["midpoint"])

    df["probability"] = df.apply(calc_prob, axis=1)

    return df[["race_id", "party", "probability", "last_trade_weight", "composite",
               "last_price", "yes_bid", "yes_ask", "midpoint", "spread",
               "volume", "open_interest", "is_stale", "has_two_sided"]]


def get_republican_win_pct(kalshi_probs):
    """
    Normalize probabilities and return Republican win % per race.
    D + R probabilities are scaled to sum to 100%.
    """
    results = []

    for race_id, group in kalshi_probs.groupby("race_id"):
        valid = group[group["probability"].notna()]

        if len(valid) == 0:
            results.append({"race_id": race_id, "kalshi": np.nan})
            continue

        # Normalize to 100%
        total = valid["probability"].sum()
        if total <= 0:
            results.append({"race_id": race_id, "kalshi": np.nan})
            continue

        normalized = valid.copy()
        normalized["prob_normalized"] = (normalized["probability"] / total) * 100

        # Get Republican probability
        rep_rows = normalized[normalized["party"] == "R"]
        if len(rep_rows) > 0:
            rep_pct = rep_rows["prob_normalized"].values[0]
        else:
            # No Republican contract - infer from Democrat (100 - D%)
            dem_rows = normalized[normalized["party"] == "D"]
            if len(dem_rows) > 0:
                rep_pct = 100 - dem_rows["prob_normalized"].values[0]
            else:
                rep_pct = np.nan

        results.append({"race_id": race_id, "kalshi": rep_pct})

    return pd.DataFrame(results)


# Run aggregation
print("Computing liquidity-weighted probabilities...")
kalshi_probs = compute_kalshi_probabilities(markets_df)
kalshi_results = get_republican_win_pct(kalshi_probs)

print(f"Races with Kalshi data: {kalshi_results['kalshi'].notna().sum()}")
print(f"Races without data: {kalshi_results['kalshi'].isna().sum()}")

Computing liquidity-weighted probabilities...
Races with Kalshi data: 449
Races without data: 55


In [ ]:
# Fill merged["kalshi"] with Republican win %
merged = merged.drop(columns=["kalshi"], errors="ignore")
merged = merged.merge(kalshi_results, on="race_id", how="left")

print(f"Races with kalshi data: {merged['kalshi'].notna().sum()} / {len(merged)}")
merged[merged["kalshi"].notna()][["race_id", "fundamentals", "cook", "kalshi"]].head(15)

Races with kalshi data: 449 / 506


,race_id,fundamentals,cook,kalshi
0,H2026AK00,3.153412,Likely R,77.120023
2,H2026AL02,-12.885976,NaN,0.000000
3,H2026AL03,37.569038,NaN,100.000000
4,H2026AL04,55.746302,NaN,100.000000
6,H2026AL06,31.624084,NaN,100.000000
7,H2026AL07,-29.108979,NaN,0.000000
9,H2026AR02,9.789692,NaN,92.835800
11,H2026AR04,33.860897,NaN,100.000000
12,H2026AZ01,-2.796439,NaN,26.438371
13,H2026AZ02,5.310723,NaN,71.842353


In [ ]:
merged.head()

,race_id,fundamentals,cook,sabato,inside,poll_avg,enop,fun_pol,fun_pol_exp,kalshi
0,H2026AK00,3.153412,Likely R,Likely R,Likely R,NaN,NaN,3.153412,5.082786,77.120023
1,H2026AL01,43.146753,NaN,NaN,NaN,NaN,NaN,43.146753,43.146753,NaN
2,H2026AL02,-12.885976,NaN,NaN,NaN,NaN,NaN,-12.885976,-12.885976,0.000000
3,H2026AL03,37.569038,NaN,NaN,NaN,NaN,NaN,37.569038,37.569038,100.000000
4,H2026AL04,55.746302,NaN,NaN,NaN,NaN,NaN,55.746302,55.746302,100.000000


In [ ]:
from statistics import NormalDist
import numpy as np
import pandas as pd

election_day = date(2026, 11, 3)
days_until_election = (election_day - date.today()).days
x = max(days_until_election, 0)
sd_house = ((x ** 0.6) / 3200.0) + 0.036

mult = {
        "H": 1.01,
        "G": 0.98,
        "S": 0.88,
        "P": 0.83,  # future, also lower H, G, S by .02 for pres cycle
    }

nd = NormalDist()
eps = 1e-4  # avoid inf when kalshi is 0 or 100

# 1) kalshi % -> probability
p = merged["kalshi"] / 100.0
p = p.clip(lower=eps, upper=1 - eps)

# 2) convert win prob to z-score (inverse normal CDF)
z = p.apply(lambda v: nd.inv_cdf(v) if pd.notna(v) else np.nan)

# 3) election-specific sd of R vote share: base sd * race-type multiplier
race_prefix = merged["race_id"].str[0]
sigma = sd_house * race_prefix.map(mult).fillna(1.0)

# 4) implied GOP margin in points (positive = GOP, negative = DEM)
# margin = z * sigma * 200  (because margin = (2*(0.5+z*sigma)-1)*100)
merged["kalshi_margin"] = (z * sigma * 200).round(2)

merged.tail(51)

0.04466463458072158


,race_id,fundamentals,cook,sabato,inside,poll_avg,enop,fun_pol,fun_pol_exp,kalshi,kalshi_margin
455,S2026NE02,9.450099,Solid R,Likely R,Solid R,NaN,NaN,9.450099,11.111077,97.567935,15.50
456,S2026NH02,-6.671561,Lean D,Lean D,Lean D,NaN,NaN,-6.671561,-6.671561,15.790684,-7.89
457,S2026NJ02,-14.622934,Solid D,Solid D,Solid D,NaN,NaN,-14.622934,-15.407366,3.774154,-13.97
458,S2026NM02,-13.677726,Solid D,Solid D,Solid D,NaN,NaN,-13.677726,-14.774076,2.208637,-15.82
459,S2026OH03,3.269313,Lean R,Lean R,Lean R,NaN,NaN,3.269313,3.510440,56.942549,1.37
460,S2026OK02,24.779351,Solid R,Solid R,Solid R,NaN,NaN,24.779351,24.779351,98.137187,16.37
461,S2026OR02,-18.293351,Solid D,Solid D,Solid D,NaN,NaN,-18.293351,-18.293351,5.460067,-12.59
462,S2026RI02,-22.482634,Solid D,Solid D,Solid D,NaN,NaN,-22.482634,-22.482634,5.092475,-12.86
463,S2026SC02,10.269631,Solid R,Solid R,Solid R,NaN,NaN,10.269631,12.490653,89.425497,9.82
464,S2026SD02,21.796934,Solid R,Solid R,Solid R,NaN,NaN,21.796934,21.796934,97.752638,15.76


In [ ]:
merged["point_estimate"] = merged["fun_pol_exp"] * .7 + merged["kalshi_margin"] * .3 ### NEED TO WEIGHT HIGHER IN COMPETITEVE RACES/LOWER IN LESS COMPETITIVE

In [ ]:
output_path = "/content/drive/MyDrive/Forecasting 2026/workflow/intermediary/output_pe.csv"
merged.to_csv(output_path, index=False)